In [4]:
# ============================================================
# RELOAD EVENTS MODULE
# ============================================================

import importlib
import src.events

importlib.reload(src.events)

from src.events import SimpleEventDetector

event_detector = SimpleEventDetector()

print("Available methods:")
print([m for m in dir(event_detector) if "flow" in m])

Available methods:
[]


In [3]:
# ============================================================
# 1. IMPORTS AND SETUP
# ============================================================

import sys
from pathlib import Path
import pandas as pd
import importlib

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

sys.path.append(str(project_root))

from src.data_pipeline import DatabaseManager, DataLoader, DataPreprocessor
import src.events
importlib.reload(src.events)

from src.events import SimpleEventDetector

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

db = DatabaseManager(db_path)
loader = DataLoader(db)
pre = DataPreprocessor()

event_detector = SimpleEventDetector()

print("Setup completed")

Setup completed


In [3]:
# ============================================================
# 2. LOAD DATA
# ============================================================

df_prices = loader.load_prices()
df_volumes = loader.load_volumes()
df_flows = loader.load_flows()
df_capacities = loader.load_capacities()
df_zones = loader.load_bidding_zones()

print("Prices:", df_prices.shape)
print("Volumes:", df_volumes.shape)
print("Flows:", df_flows.shape)
print("Capacities:", df_capacities.shape)
print("Zones:", df_zones.shape)

Prices: (1007980, 5)
Volumes: (350880, 6)
Flows: (230470, 6)
Capacities: (35132, 7)
Zones: (20, 4)


In [4]:
# ============================================================
# 3. CREATE DATETIME INDEX
# ============================================================

prices_dt = pre.create_datetime_index(df_prices, convert_to_utc=False)
volumes_dt = pre.create_datetime_index(df_volumes, convert_to_utc=False)
flows_dt = pre.create_datetime_index(df_flows, convert_to_utc=False)
capacities_dt = pre.create_datetime_index(df_capacities, convert_to_utc=False)

print("Prices:", prices_dt.index.min(), "->", prices_dt.index.max())
print("Volumes:", volumes_dt.index.min(), "->", volumes_dt.index.max())
print("Flows:", flows_dt.index.min(), "->", flows_dt.index.max())
print("Capacities:", capacities_dt.index.min(), "->", capacities_dt.index.max())

Prices: 2020-01-01 00:00:00 -> 2025-09-30 23:00:00
Volumes: 2020-01-01 00:00:00 -> 2021-12-31 23:00:00
Flows: 2020-01-01 00:00:00 -> 2020-12-31 23:00:00
Capacities: 2020-01-01 00:00:00 -> 2020-12-31 23:00:00


In [5]:
# ============================================================
# 4. DETECT PRICE EVENTS
# ============================================================

df_price_events = event_detector.detect_price_events(
    prices_dt.reset_index()
)

price_event_columns = [
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility"
]

display(df_price_events.head())

price_event_counts = (
    df_price_events[price_event_columns]
    .sum()
    .reset_index()
)

price_event_counts.columns = ["event_type", "count"]

display(price_event_counts)

,datetime,price_id,zone_id,delivery_day,hour,price_value,price_delta,abs_price_delta,rolling_volatility_24h,low_price,high_price,price_spike,extreme_price,rapid_price_change,price_ramp_up,price_ramp_down,high_volatility
0,2020-01-01 00:00:00,1,1,2020-01-01,0,28.78,NaN,NaN,NaN,False,False,False,False,False,False,False,False
25,2020-01-01 01:00:00,21,1,2020-01-01,1,28.45,-0.33,0.33,NaN,False,False,False,False,False,False,False,False
41,2020-01-01 02:00:00,41,1,2020-01-01,2,27.90,-0.55,0.55,NaN,False,False,False,False,False,False,False,False
63,2020-01-01 03:00:00,61,1,2020-01-01,3,27.52,-0.38,0.38,NaN,False,False,False,False,False,False,False,False
89,2020-01-01 04:00:00,81,1,2020-01-01,4,27.54,0.02,0.02,NaN,False,False,False,False,False,False,False,False


,event_type,count
0,low_price,100714
1,high_price,100790
2,price_spike,50385
3,extreme_price,10077
4,rapid_price_change,50394
5,price_ramp_up,50395
6,price_ramp_down,50389
7,high_volatility,100800


In [6]:
# ============================================================
# 5. DETECT VOLUME EVENTS
# ============================================================

df_volume_events = event_detector.detect_volume_events(
    volumes_dt.reset_index()
)

volume_event_columns = [
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_spike",
    "sell_volume_spike"
]

display(df_volume_events.head())

volume_event_counts = (
    df_volume_events[volume_event_columns]
    .sum()
    .reset_index()
)

volume_event_counts.columns = ["event_type", "count"]

display(volume_event_counts)

,datetime,volume_id,zone_id,delivery_day,hour,buy_volume_value,sell_volume_value,high_demand,low_demand,high_generation,low_generation,volume_imbalance,strong_buy_pressure,strong_sell_pressure,buy_volume_delta,sell_volume_delta,abs_buy_volume_delta,abs_sell_volume_delta,buy_volume_spike,sell_volume_spike
0,2020-01-01 00:00:00,1,1,2020-01-01,0,688.8,283.8,False,False,False,False,0.416410,False,False,NaN,NaN,NaN,NaN,False,False
26,2020-01-01 01:00:00,21,1,2020-01-01,1,680.7,284.8,False,False,False,False,0.410047,False,False,-8.1,1.0,8.1,1.0,False,False
43,2020-01-01 02:00:00,41,1,2020-01-01,2,669.8,281.2,False,False,False,False,0.408623,False,False,-10.9,-3.6,10.9,3.6,False,False
61,2020-01-01 03:00:00,61,1,2020-01-01,3,666.4,266.2,False,False,False,False,0.429123,False,False,-3.4,-15.0,3.4,15.0,False,False
80,2020-01-01 04:00:00,81,1,2020-01-01,4,663.0,252.6,False,False,False,False,0.448231,False,False,-3.4,-13.6,3.4,13.6,False,False


,event_type,count
0,high_demand,35044
1,low_demand,35064
2,high_generation,35090
3,low_generation,33340
4,strong_buy_pressure,35096
5,strong_sell_pressure,35091
6,buy_volume_spike,17450
7,sell_volume_spike,17507


In [7]:
# ============================================================
# 6. DETECT FLOW EVENTS
# ============================================================

df_flow_events = event_detector.detect_flow_events(
    flows_dt.reset_index()
)

flow_event_columns = [
    "low_flow",
    "high_flow",
    "flow_spike",
    "flow_reversal",
    "has_flow_event"
]

display(df_flow_events.head())

flow_event_counts = (
    df_flow_events[flow_event_columns]
    .sum()
    .reset_index()
)

flow_event_counts.columns = ["event_type", "count"]

display(flow_event_counts)

AttributeError: 'SimpleEventDetector' object has no attribute 'detect_flow_events'